In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import os, random
import numpy as np
import open3d as o3d
from pathlib import Path

# OCC
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.BRepAdaptor import BRepAdaptor_Surface, BRepAdaptor_Curve
from OCC.Core.TopExp import TopExp_Explorer, topexp 
from OCC.Core.TopTools import TopTools_IndexedMapOfShape
from OCC.Core.TopAbs import TopAbs_FACE, TopAbs_EDGE, TopAbs_VERTEX, TopAbs_FORWARD, TopAbs_REVERSED
from OCC.Core.BRep import BRep_Tool
from OCC.Core.BRepTools import breptools
import OCC.Core.gp as gp

from OCC.Core.GeomAbs import (
    GeomAbs_Plane, GeomAbs_Cylinder, GeomAbs_Cone, 
    GeomAbs_Sphere, GeomAbs_Torus, GeomAbs_BezierSurface, 
    GeomAbs_BSplineSurface, GeomAbs_SurfaceOfRevolution, 
    GeomAbs_SurfaceOfExtrusion, GeomAbs_OffsetSurface, 
    GeomAbs_OtherSurface,
    GeomAbs_Circle, GeomAbs_Ellipse, GeomAbs_BSplineCurve,
    # GeomAbs_Hyperbola, GeomAbs_Parabola,
    # GeomAbs_BezierCurve, GeomAbs_Line
)

# B-Rep visualization utilities
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from OCC.Core.GCPnts import GCPnts_QuasiUniformDeflection
from OCC.Core.TopLoc import TopLoc_Location
from OCC.Core.Geom import Geom_Plane, Geom_CylindricalSurface, Geom_ConicalSurface, Geom_SphericalSurface, Geom_ToroidalSurface
from OCC.Core.GeomAPI import GeomAPI_ProjectPointOnSurf
from OCC.Core.gp import gp_Pnt
from shapely.geometry import Point, Polygon
from shapely.ops import polygonize, linemerge, unary_union
import math

COLOR_MAP = {
    GeomAbs_Plane: [0.8, 0.8, 0.8],               
    GeomAbs_Cylinder: [0.2, 0.6, 1.0],            
    GeomAbs_Cone: [1.0, 0.8, 0.0],                
    GeomAbs_Sphere: [0.2, 0.8, 0.2],              
    GeomAbs_Torus: [1.0, 0.5, 0.0],               
    GeomAbs_BSplineSurface: [1.0, 0.2, 0.2],      
    GeomAbs_SurfaceOfRevolution: [0.6, 0.2, 0.8], 
    GeomAbs_SurfaceOfExtrusion: [0.4, 0.8, 0.8],
    GeomAbs_BezierSurface: [1.0, 0.4, 0.7],       
    GeomAbs_OffsetSurface: [0.5, 0.5, 0.0],       
    GeomAbs_OtherSurface: [0.3, 0.3, 0.3]         
}
DEFAULT_COLOR = [1.0, 1.0, 1.0]

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
class ABCMultiModalDataset(Dataset):
    def __init__(self, base_dir, pcd_num_points=2048, voxel_res=64, 
                 max_vertices=4000, max_edges=2000, max_faces=1000):
        self.base_dir = Path(base_dir)
        self.pcd_num_points = pcd_num_points
        self.voxel_res = voxel_res
        self.max_vertices = max_vertices
        self.max_edges = max_edges
        self.max_faces = max_faces
        self.device = o3d.core.Device("CPU:0")
        
        self.obj_files = []
        print("OBJ file scanning...")
        for chunk_folder in sorted(os.listdir(self.base_dir)):
            chunk_path = self.base_dir / chunk_folder
            if chunk_path.is_dir():
                for model_id in os.listdir(chunk_path):
                    model_path = chunk_path / model_id
                    objs = list(model_path.glob("*.obj"))
                    if objs:
                        self.obj_files.append(objs[0])
        print(f"{len(self.obj_files)} OBJ files loaded.")

    def __len__(self):
        return len(self.obj_files)

    def _extract_complex_brep_target(self, step_path):
        """Extract B-rep targets based on ISO-10303-42."""
        reader = STEPControl_Reader()
        if reader.ReadFile(str(step_path)) != 1:
            return None 
            
        reader.TransferRoots()
        shape = reader.OneShape()
        
# 1. Build unique topology maps (V/E/F)
        map_V = TopTools_IndexedMapOfShape()
        map_E = TopTools_IndexedMapOfShape()
        map_F = TopTools_IndexedMapOfShape()
        
# Resolve deprecation warning: use static topexp.MapShapes
        topexp.MapShapes(shape, TopAbs_VERTEX, map_V) 
        topexp.MapShapes(shape, TopAbs_EDGE, map_E)   
        topexp.MapShapes(shape, TopAbs_FACE, map_F)   
        
        num_V, num_E, num_F = map_V.Size(), map_E.Size(), map_F.Size()
        
# [Translated] See surrounding code for details.
        V_feat = np.zeros((self.max_vertices, 3))
        E_feat = np.zeros((self.max_edges, 73)) # translated comment
                                                # Sample up to 10 B-spline control points
                                                # [Translated] See surrounding code for details.
        F_feat = np.zeros((self.max_faces, 174)) # translated comment
                                                 # Sample up to 25 B-spline control points (5x5)
                                                 # [Translated] See surrounding code for details.
                                                 # [Translated] See surrounding code for details.
        
# Initialize adjacency matrices
        adj_EV = np.zeros((self.max_edges, self.max_vertices)) 
        adj_FE = np.zeros((self.max_faces, self.max_edges))    
        
        # Vertex processing
        for i in range(1, num_V + 1):
            if i > self.max_vertices: break
            v_shape = map_V.FindKey(i)
            pnt = BRep_Tool.Pnt(v_shape) 
            V_feat[i-1] = [pnt.X(), pnt.Y(), pnt.Z()] 
            
# Edge processing
        for i in range(1, num_E + 1):
            if i > self.max_edges: break
            e_shape = map_E.FindKey(i)
            
# Edge-vertex incidence and orientation
            v_first, v_last = topexp.FirstVertex(e_shape), topexp.LastVertex(e_shape)
            if not v_first.IsNull():
                idx_v1 = map_V.FindIndex(v_first)
                if idx_v1 <= self.max_vertices: adj_EV[i-1, idx_v1-1] = 1.0  
            if not v_last.IsNull():
                idx_v2 = map_V.FindIndex(v_last)
                if idx_v2 <= self.max_vertices: adj_EV[i-1, idx_v2-1] = -1.0 
                
            try: # translated comment
                adaptor = BRepAdaptor_Curve(e_shape)
                tmin, tmax = adaptor.FirstParameter(), adaptor.LastParameter()
                ctype = adaptor.GetType()
                
                p_start, p_end = adaptor.Value(tmin), adaptor.Value(tmax)
                ori = 0.0 if e_shape.Orientation() == TopAbs_FORWARD.value else 1.0 

                # [Translated] See surrounding code for details.
                E_feat[i-1, 0:9] = [float(ctype), tmin, tmax, p_start.X(), p_start.Y(), p_start.Z(), p_end.X(), p_end.Y(), p_end.Z()]
                E_feat[i-1, 11] = ori

                # [Translated] See surrounding code for details.
                if ctype == GeomAbs_Circle: E_feat[i-1, 9] = adaptor.Circle().Radius() 
                elif ctype == GeomAbs_Ellipse:
                    E_feat[i-1, 9] = adaptor.Ellipse().MajorRadius()
                    E_feat[i-1, 10] = adaptor.Ellipse().MinorRadius()
                elif ctype == GeomAbs_BSplineCurve:
                    b_curve = adaptor.BSpline()
                    E_feat[i-1, 9] = float(b_curve.Degree())
                    nb_poles = b_curve.NbPoles()
                    E_feat[i-1, 12] = float(nb_poles) 
                    # [Translated] See surrounding code for details.
                    p_indices = np.linspace(1, nb_poles, min(10, nb_poles))
                    pole_idx = 13
                    for p_i in p_indices: 
                        pole = b_curve.Pole(int(round(p_i)))
                        E_feat[i-1, pole_idx:pole_idx+3] = [pole.X(), pole.Y(), pole.Z()]
                        pole_idx += 3

                # [Translated] See surrounding code for details.
                pts_idx = 43
                for t in np.linspace(tmin, tmax, 10):
                    pt = adaptor.Value(t)
                    E_feat[i-1, pts_idx:pts_idx+3] = [pt.X(), pt.Y(), pt.Z()]
                    pts_idx += 3

            except Exception as e:
                print(e)
                pass
                
# Face processing
        for i in range(1, num_F + 1):
            if i > self.max_faces: break
            f_shape = map_F.FindKey(i)
            
# Face-edge incidence (bound loop traversal)
            exp_e = TopExp_Explorer(f_shape, TopAbs_EDGE)
            while exp_e.More():
                e_in_f = exp_e.Current()
                idx_e = map_E.FindIndex(e_in_f)
                if 0 < idx_e <= self.max_edges:
                    # 0 is reserved for 'not connected', so keep orientation as +1 / -1.
                    e_ori = 1.0 if e_in_f.Orientation() == TopAbs_FORWARD.value else -1.0
                    adj_FE[i-1, idx_e-1] = e_ori 
                exp_e.Next()
                
            try:
                # Face Type
                adaptor = BRepAdaptor_Surface(f_shape)
                umin, umax, vmin, vmax = breptools.UVBounds(f_shape)
                # umin, umax = adaptor.FirstUParameter(), adaptor.LastUParameter()
                # vmin, vmax = adaptor.FirstVParameter(), adaptor.LastVParameter()
                stype = adaptor.GetType()
                
                # Face Param
                umid, vmid = (umin + umax) / 2.0, (vmin + vmax) / 2.0
                p_mid = adaptor.Value(umid, vmid)
                
                # Extract primitive local frame (origin + axes)
                loc_x, loc_y, loc_z = p_mid.X(), p_mid.Y(), p_mid.Z() # default location
                z_dir = [0, 0, 1] # default Z axis
                x_dir = [1, 0, 0] # default X axis

                # Extract primitive local frame (origin + axes)
                pos = None
                if stype == GeomAbs_Plane: pos = adaptor.Plane().Position()
                elif stype == GeomAbs_Cylinder: pos = adaptor.Cylinder().Position()
                elif stype == GeomAbs_Cone: pos = adaptor.Cone().Position()
                elif stype == GeomAbs_Sphere: pos = adaptor.Sphere().Position()
                elif stype == GeomAbs_Torus: pos = adaptor.Torus().Position()

                if pos is not None:
                    loc = pos.Location()
                    z_axis = pos.Direction()
                    x_axis = pos.XDirection()
                    loc_x, loc_y, loc_z = loc.X(), loc.Y(), loc.Z()
                    z_dir = [z_axis.X(), z_axis.Y(), z_axis.Z()]
                    x_dir = [x_axis.X(), x_axis.Y(), x_axis.Z()]

                P = gp.gp_Pnt()
                V1U, V1V = gp.gp_Vec(), gp.gp_Vec()
                adaptor.D1(umid, vmid, P, V1U, V1V) # first derivative
                normal = V1U.Crossed(V1V) # normal from cross product
                if normal.Magnitude() > 1e-10:
                    normal.Normalize()
                else:
                    normal = gp.gp_Vec(0, 0, 1) # fallback normal
                nx, ny, nz = normal.X(), normal.Y(), normal.Z()

                ori = 0.0 if f_shape.Orientation() == TopAbs_FORWARD.value else 1.0 
                
                F_feat[i-1, 0:8] = [float(stype), umin, umax, vmin, vmax, p_mid.X(), p_mid.Y(), p_mid.Z()]
                F_feat[i-1, 8:11] = [nx, ny, nz]
                F_feat[i-1, 13] = ori

                if stype == GeomAbs_Cylinder: F_feat[i-1, 11] = adaptor.Cylinder().Radius() 
                elif stype == GeomAbs_Sphere: F_feat[i-1, 11] = adaptor.Sphere().Radius() 
                elif stype == GeomAbs_Cone:
                    F_feat[i-1, 11] = adaptor.Cone().RefRadius() 
                    F_feat[i-1, 12] = adaptor.Cone().SemiAngle()
                elif stype == GeomAbs_Torus:
                    F_feat[i-1, 11] = adaptor.Torus().MajorRadius()
                    F_feat[i-1, 12] = adaptor.Torus().MinorRadius()
                elif stype == GeomAbs_BSplineSurface:
                    bspl = adaptor.BSpline()
                    F_feat[i-1, 11] = float(bspl.UDegree())
                    F_feat[i-1, 12] = float(bspl.VDegree())
                    nb_u, nb_v = bspl.NbUPoles(), bspl.NbVPoles()
                    F_feat[i-1, 14] = float(nb_u * nb_v) 

                    # [Translated] See surrounding code for details.
                    u_indices = np.linspace(1, nb_u, min(5, nb_u))
                    v_indices = np.linspace(1, nb_v, min(5, nb_v))
                    pole_idx = 15
                    for u in u_indices:
                        for v in v_indices:
                            pole = bspl.Pole(int(round(u)), int(round(v)))
                            F_feat[i-1, pole_idx:pole_idx+3] = [pole.X(), pole.Y(), pole.Z()]
                            pole_idx += 3

                # [Translated] See surrounding code for details.
                F_feat[i-1, 90:93] = [loc_x, loc_y, loc_z]
                F_feat[i-1, 93:96] = z_dir
                F_feat[i-1, 96:99] = x_dir

                # [Update] Sample UV-grid points (5x5)
                grid_idx = 99
                for u in np.linspace(umin, umax, 5):
                    for v in np.linspace(vmin, vmax, 5):
                        pt = adaptor.Value(u, v)
                        F_feat[i-1, grid_idx:grid_idx+3] = [pt.X(), pt.Y(), pt.Z()]
                        grid_idx += 3

            except Exception as e:
                print(e)
                pass

        return {
            "v_feat": torch.tensor(V_feat).float(),
            "e_feat": torch.tensor(E_feat).float(),
            "f_feat": torch.tensor(F_feat).float(),
            "adj_ev": torch.tensor(adj_EV).float(),
            "adj_fe": torch.tensor(adj_FE).float(),
            "counts": torch.tensor([num_V, num_E, num_F]).int() 
        }

    # stereo depth camera raycasting simul.
    def _simulate_stereo_depth(self, mesh, max_mesh_dist):
        dist_min = max_mesh_dist * 2.5
        dist_max = max_mesh_dist * 4.0
        
        phi, theta = np.random.uniform(0, 2*np.pi), np.arccos(np.random.uniform(-1, 1))
        radius = np.random.uniform(dist_min, dist_max)
        
        cam_pos = np.array([
            radius * np.sin(theta) * np.cos(phi),
            radius * np.sin(theta) * np.sin(phi),
            radius * np.cos(theta)
        ])
        
        scene = o3d.t.geometry.RaycastingScene()
        scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh, device=self.device))
        
        rays = scene.create_rays_pinhole(fov_deg=69, center=[0,0,0], eye=cam_pos, up=[0,1,0], width_px=320, height_px=240)
        ans = scene.cast_rays(rays)
        t_hit = ans['t_hit'].numpy()
        hit_mask, rays_arr = np.isfinite(t_hit), rays.numpy()
        pts = rays_arr[hit_mask][:, :3] + t_hit[hit_mask][:, np.newaxis] * rays_arr[hit_mask][:, 3:]
        
        z_depths = np.dot(pts - cam_pos, -cam_pos/np.linalg.norm(cam_pos))
        noise = np.random.normal(0, 1, size=pts.shape) * ((z_depths**2 * 0.08) / (800 * 50))[:, np.newaxis]
        return pts + noise

    def __getitem__(self, idx):
        obj_path = self.obj_files[idx]
        step_path = obj_path.with_suffix('.step')
        
        mesh = o3d.io.read_triangle_mesh(str(obj_path))
        mesh.translate(-mesh.get_center())
        max_mesh_dist = np.max(np.linalg.norm(np.asarray(mesh.vertices), axis=1))
        
        raw_sim_points = self._simulate_stereo_depth(mesh, max_mesh_dist)
        pcd_pts = raw_sim_points / (max_mesh_dist + 1e-8)
        
        if len(pcd_pts) >= self.pcd_num_points:
            pcd_pts = pcd_pts[np.random.choice(len(pcd_pts), self.pcd_num_points, replace=False)]
        else:
            pcd_pts = pcd_pts[np.random.choice(len(pcd_pts), self.pcd_num_points, replace=True)]
        
        voxel_pts = (pcd_pts * 0.5) + 0.5 
        voxel_indices = np.clip(((voxel_pts * (self.voxel_res - 1)).astype(int)), 0, self.voxel_res - 1)
        voxel_matrix = np.zeros((self.voxel_res, self.voxel_res, self.voxel_res), dtype=np.float32)
        voxel_matrix[voxel_indices[:, 0], voxel_indices[:, 1], voxel_indices[:, 2]] = 1.0

        brep_target = self._extract_complex_brep_target(step_path)
        
        if brep_target is None:
            brep_target = {
                "v_feat": torch.zeros((self.max_vertices, 3)), "e_feat": torch.zeros((self.max_edges, 12)),
                "f_feat": torch.zeros((self.max_faces, 14)), "adj_ev": torch.zeros((self.max_edges, self.max_vertices)),
                "adj_fe": torch.zeros((self.max_faces, self.max_edges)), "counts": torch.zeros(3).int()
            }
        
        return {
            "pcd": torch.from_numpy(pcd_pts).float(),
            "voxel": torch.from_numpy(voxel_matrix).float().unsqueeze(0),
            **brep_target, 
            "model_id": obj_path.parent.name,
            "max_mesh_dist": max_mesh_dist,
            "obj_path": obj_path,
            "step_path": step_path
        }

In [3]:
dataset = ABCMultiModalDataset(base_dir="./abc_dataset_filtered-1", pcd_num_points=2048, voxel_res=64)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=4)

OBJ file scanning...
121146 OBJ files loaded.


### Sample Data Visualization: PCD + OBJ / B-Rep (STEP) / Voxel

In [4]:
sample = dataset[random.randint(0, len(dataset)-1)]
# sample = dataset[10]

obj_path = Path(sample['obj_path'])
step_path = Path(sample['step_path'])
max_dist = sample['max_mesh_dist']

print(f"Visualizing Model ID: {sample['model_id']}")

# Original OBJ mesh visualization
orig_mesh = o3d.io.read_triangle_mesh(str(obj_path))
orig_mesh.translate(-orig_mesh.get_center())
orig_mesh.scale(1.0 / max_dist, center=(0, 0, 0))
orig_mesh.paint_uniform_color([0.7, 0.7, 0.7])
orig_mesh.translate([-2, 0, 0]) # shift left

# Point-cloud visualization data
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(sample['pcd'].numpy())
pcd.paint_uniform_color([1, 0, 0])
pcd.translate([-2, 0, 0]) # shift left

# Voxel visualization data
vox_matrix = sample['voxel'].squeeze().numpy()
indices = np.argwhere(vox_matrix == 1.0)
v_pts = (indices / (dataset.voxel_res - 1) - 0.5) * 2.0
voxel_pcd = o3d.geometry.PointCloud()
voxel_pcd.points = o3d.utility.Vector3dVector(v_pts)
voxel_pcd.paint_uniform_color([0, 1, 0])
voxel_pcd.translate([2, 0, 0]) # shift right

# [Translated] See surrounding code for details.
reader = STEPControl_Reader()
reader.ReadFile(str(step_path))
reader.TransferRoots()
shape = reader.OneShape()
BRepMesh_IncrementalMesh(shape, 0.05)

# Face processing
step_combined_mesh = o3d.geometry.TriangleMesh()
explorer_face = TopExp_Explorer(shape, TopAbs_FACE)
while explorer_face.More():
    face = explorer_face.Current()
    loc = TopLoc_Location()
    triangulation = BRep_Tool.Triangulation(face, loc)
    if triangulation:
        adaptor = BRepAdaptor_Surface(face)
        surf_type = adaptor.GetType()
        face_color = COLOR_MAP.get(surf_type, DEFAULT_COLOR)
        
        verts = []
        for i in range(1, triangulation.NbNodes() + 1):
            p = triangulation.Node(i).Transformed(loc.Transformation())
            verts.append([p.X(), p.Y(), p.Z()])
            
        tris = []
        is_reversed = (face.Orientation() == TopAbs_REVERSED)
        for i in range(1, triangulation.NbTriangles() + 1):
            t = triangulation.Triangle(i)
            n1, n2, n3 = t.Get()
            if is_reversed: tris.append([n1-1, n3-1, n2-1])
            else: tris.append([n1-1, n2-1, n3-1])
            
        f_mesh = o3d.geometry.TriangleMesh()
        f_mesh.vertices = o3d.utility.Vector3dVector(np.array(verts))
        f_mesh.triangles = o3d.utility.Vector3iVector(np.array(tris))
        f_mesh.paint_uniform_color(face_color)
        step_combined_mesh += f_mesh
    explorer_face.Next()

# Edge processing
edge_points, edge_lines = [], []
point_idx = 0
explorer_edge = TopExp_Explorer(shape, TopAbs_EDGE)
while explorer_edge.More():
    edge = explorer_edge.Current()
    adaptor = BRepAdaptor_Curve(edge)
    discretizer = GCPnts_QuasiUniformDeflection(adaptor, 0.01)
    if discretizer.IsDone():
        pts = [[discretizer.Value(i).X(), discretizer.Value(i).Y(), discretizer.Value(i).Z()] 
               for i in range(1, discretizer.NbPoints() + 1)]
        if len(pts) > 1:
            edge_points.extend(pts)
            for i in range(len(pts) - 1):
                edge_lines.append([point_idx + i, point_idx + i + 1])
            point_idx += len(pts)
    explorer_edge.Next()

wireframe = o3d.geometry.LineSet()
wireframe.points = o3d.utility.Vector3dVector(np.array(edge_points))
wireframe.lines = o3d.utility.Vector2iVector(np.array(edge_lines))
wireframe.paint_uniform_color([0, 0, 0]) # black wireframe

# Point-cloud visualization data
# Original OBJ mesh visualization
# [Translated] See surrounding code for details.
step_center = step_combined_mesh.get_center()
step_combined_mesh.translate(-step_center)
step_combined_mesh.scale(1.0 / max_dist, center=(0, 0, 0))
wireframe.translate(-step_center)
wireframe.scale(1.0 / max_dist, center=(0, 0, 0))

# Point-cloud visualization data
# [Translated] See surrounding code for details.
step_combined_mesh.translate([0, 0, 0])
wireframe.translate([0, 0, 0])

# [Translated] See surrounding code for details.
print("Rendered:")
print(" - LEFT  : STEP B-Rep Class (Primitive Color + Black Wireframe)")
print(" - MIDDLE: OBJ Mesh(Grey) + Sim. PCD(Red) overlay")
print(" - RIGHT : Sim. Voxel Grid(Green)")

o3d.visualization.draw_geometries(
    [orig_mesh, pcd, voxel_pcd, step_combined_mesh, wireframe],
    window_name=f"Comprehensive CAD Data Check: {sample['model_id']}",
    width=1500, height=1000,
    mesh_show_back_face=True
)

Visualizing Model ID: 00021005
Rendered:
 - LEFT  : STEP B-Rep Class (Primitive Color + Black Wireframe)
 - MIDDLE: OBJ Mesh(Grey) + Sim. PCD(Red) overlay
 - RIGHT : Sim. Voxel Grid(Green)


### Sample Data Visualization: Original vs Extracted B-Rep

In [10]:
# ==========================================
# [Translated] See surrounding code for details.
# ==========================================
sample = dataset[random.randint(0, len(dataset)-1)]
# sample = dataset[301]

step_path = str(sample['step_path'])
max_dist = sample['max_mesh_dist']

v_feat = sample['v_feat'].numpy()
e_feat = sample['e_feat'].numpy()
f_feat = sample['f_feat'].numpy()
adj_FE = sample['adj_fe'].numpy()
adj_EV = sample['adj_ev'].numpy()

num_V = min(int(sample['counts'][0]), v_feat.shape[0])
num_E = min(int(sample['counts'][1]), e_feat.shape[0])
num_F = min(int(sample['counts'][2]), f_feat.shape[0])

print(f"Visualizing Model ID: {sample['model_id']}")

# ==========================================
# Wireframe processing
# ==========================================
reader = STEPControl_Reader()
reader.ReadFile(step_path)
reader.TransferRoots()
shape = reader.OneShape()
BRepMesh_IncrementalMesh(shape, 0.05)

# Face processing
orig_mesh = o3d.geometry.TriangleMesh()
explorer_face = TopExp_Explorer(shape, TopAbs_FACE)
while explorer_face.More():
    face = explorer_face.Current()
    loc = TopLoc_Location()
    triangulation = BRep_Tool.Triangulation(face, loc)
    if triangulation:
        adaptor = BRepAdaptor_Surface(face)
        surf_type = adaptor.GetType()
        face_color = COLOR_MAP.get(surf_type, DEFAULT_COLOR)
        
        verts, tris = [], []
        for i in range(1, triangulation.NbNodes() + 1):
            p = triangulation.Node(i).Transformed(loc.Transformation())
            verts.append([p.X(), p.Y(), p.Z()])
        
        is_rev = (face.Orientation() == TopAbs_REVERSED)
        for i in range(1, triangulation.NbTriangles() + 1):
            t = triangulation.Triangle(i)
            n1, n2, n3 = t.Get()
            if is_rev: tris.append([n1-1, n3-1, n2-1])
            else: tris.append([n1-1, n2-1, n3-1])
            
        f_mesh = o3d.geometry.TriangleMesh()
        f_mesh.vertices = o3d.utility.Vector3dVector(np.array(verts, dtype=np.float64))
        f_mesh.triangles = o3d.utility.Vector3iVector(np.array(tris, dtype=np.int32))
        f_mesh.paint_uniform_color(face_color)
        orig_mesh += f_mesh
    explorer_face.Next()

# Edge processing
edge_points, edge_lines = [], []
point_idx = 0
explorer_edge = TopExp_Explorer(shape, TopAbs_EDGE)
while explorer_edge.More():
    edge = explorer_edge.Current()
    adaptor = BRepAdaptor_Curve(edge)
    discretizer = GCPnts_QuasiUniformDeflection(adaptor, 0.01)
    if discretizer.IsDone():
        pts = [[discretizer.Value(i).X(), discretizer.Value(i).Y(), discretizer.Value(i).Z()] 
               for i in range(1, discretizer.NbPoints() + 1)]
        if len(pts) > 1:
            edge_points.extend(pts)
            for i in range(len(pts) - 1):
                edge_lines.append([point_idx + i, point_idx + i + 1])
            point_idx += len(pts)
    explorer_edge.Next()

orig_wireframe = o3d.geometry.LineSet()
orig_wireframe.points = o3d.utility.Vector3dVector(np.array(edge_points, dtype=np.float64))
orig_wireframe.lines = o3d.utility.Vector2iVector(np.array(edge_lines, dtype=np.int32))
orig_wireframe.paint_uniform_color([0, 0, 0])

# [Translated] See surrounding code for details.
orig_center = orig_mesh.get_center()
orig_mesh.translate(-orig_center)
orig_mesh.scale(1.0 / max_dist, center=(0, 0, 0))
orig_wireframe.translate(-orig_center)
orig_wireframe.scale(1.0 / max_dist, center=(0, 0, 0))

# ==========================================
# Wireframe processing
# ==========================================
offset = np.array([2.5, 0.0, 0.0]) # shift right

def normalize_pts(pts):
    return (pts - orig_center) / max_dist

# Edge processing
recon_e_pts, recon_e_lines = [], []
for i in range(num_E):
    pts_10 = e_feat[i, 43:73].reshape(10, 3) 
    if np.all(pts_10 == 0): continue
        
    start_idx = len(recon_e_pts)
    recon_e_pts.extend(pts_10)
    for j in range(9): 
        recon_e_lines.append([start_idx + j, start_idx + j + 1])

recon_wireframe = o3d.geometry.LineSet()
if recon_e_pts:
    recon_wireframe.points = o3d.utility.Vector3dVector(normalize_pts(np.array(recon_e_pts)))
    recon_wireframe.lines = o3d.utility.Vector2iVector(np.array(recon_e_lines))
    recon_wireframe.paint_uniform_color([0, 0, 0]) # black wireframe
    recon_wireframe.translate(offset)

# Face processing
recon_mesh = o3d.geometry.TriangleMesh()

for i in range(num_F):
    umin, umax, vmin, vmax = f_feat[i, 1:5]
    
    if abs(umax - umin) < 1e-5 or abs(vmax - vmin) < 1e-5 or abs(umax) > 1e6: 
        continue 
        
    stype = int(f_feat[i, 0])
    p1, p2 = float(f_feat[i, 11]), float(f_feat[i, 12])
    
    # [Translated] See surrounding code for details.
    matched_enum = None
    for k in COLOR_MAP.keys():
        if int(k) == stype:
            matched_enum = k
            break
            
    face_color = [0.0, 0.0, 0.0] 
    if 'COLOR_MAP' in globals():
        for k, v in COLOR_MAP.items():
            if int(k) == stype:
                face_color = v
                break

    # [Translated] See surrounding code for details.
    is_reversed = False
    if float(f_feat[i, 13]) == 1.0:
        is_reversed = True

    # [Translated] See surrounding code for details.
    if matched_enum in [GeomAbs_BezierSurface, GeomAbs_BSplineSurface, GeomAbs_SurfaceOfRevolution, GeomAbs_SurfaceOfExtrusion]: 
        grid_pts = f_feat[i, 99:174].reshape(25, 3)
        if not np.all(grid_pts == 0):
            f_tris = []
            for r in range(4): 
                for c in range(4): 
                    idx1 = r * 5 + c
                    idx2 = idx1 + 5
                    idx3 = idx1 + 1
                    idx4 = idx2 + 1
                    if is_reversed: f_tris.extend([[idx1, idx2, idx3], [idx2, idx4, idx3]])
                    else: f_tris.extend([[idx1, idx3, idx2], [idx2, idx3, idx4]])
            
            tmp_mesh = o3d.geometry.TriangleMesh()
            tmp_mesh.vertices = o3d.utility.Vector3dVector(normalize_pts(grid_pts))
            tmp_mesh.triangles = o3d.utility.Vector3iVector(np.array(f_tris))
            tmp_mesh.paint_uniform_color(face_color)
            tmp_mesh.translate(offset)
            tmp_mesh.compute_vertex_normals()
            recon_mesh += tmp_mesh
        continue 

    # Primitive ?????n
    loc = gp.gp_Pnt(float(f_feat[i, 90]), float(f_feat[i, 91]), float(f_feat[i, 92]))
    z_vec = gp.gp_Vec(float(f_feat[i, 93]), float(f_feat[i, 94]), float(f_feat[i, 95]))
    x_vec = gp.gp_Vec(float(f_feat[i, 96]), float(f_feat[i, 97]), float(f_feat[i, 98]))
    
    ax3 = None
    if z_vec.Magnitude() > 1e-5 and x_vec.Magnitude() > 1e-5:
        z_dir = gp.gp_Dir(z_vec)
        y_vec = z_vec.Crossed(x_vec)
        if y_vec.Magnitude() > 1e-5:
            ax3 = gp.gp_Ax3(loc, z_dir, gp.gp_Dir(y_vec.Crossed(z_vec)))
        else: ax3 = gp.gp_Ax3(loc, z_dir)
    else: ax3 = gp.gp_Ax3(loc, gp.gp_Dir(0,0,1))
    
    geom_surf = None
    if matched_enum == GeomAbs_Plane: geom_surf = Geom_Plane(ax3)
    elif matched_enum == GeomAbs_Cylinder and p1 >= 0: geom_surf = Geom_CylindricalSurface(ax3, p1) 
    elif matched_enum == GeomAbs_Cone and p1 >= 0: geom_surf = Geom_ConicalSurface(ax3, p2, p1) 
    elif matched_enum == GeomAbs_Sphere and p1 >= 0: geom_surf = Geom_SphericalSurface(ax3, p1) 
    elif matched_enum == GeomAbs_Torus and p1 >= 0: geom_surf = Geom_ToroidalSurface(ax3, p1, p2) 
    
    if geom_surf is not None:
        connected_edges = np.where(adj_FE[i] != 0)[0]
        
        # Vertex processing
        vertex_uv_map = {}
        for e_idx in connected_edges:
            v_indices = np.where(adj_EV[e_idx] != 0)[0]
            for v_idx in v_indices:
                if v_idx not in vertex_uv_map:
                    v_pt = v_feat[v_idx, 0:3]
                    if np.all(v_pt == 0): continue
                    proj = GeomAPI_ProjectPointOnSurf(gp_Pnt(float(v_pt[0]), float(v_pt[1]), float(v_pt[2])), geom_surf)
                    if proj.NbPoints() > 0:
                        u, v = proj.LowerDistanceParameters()
                        # [Translated] See surrounding code for details.
                        if matched_enum in [GeomAbs_Cylinder, GeomAbs_Cone, GeomAbs_Sphere, GeomAbs_Torus]:
                            period = 2 * math.pi
                            while u < umin - 0.1: u += period
                            while u > umax + 0.1: u -= period
                        vertex_uv_map[v_idx] = (round(u, 4), round(v, 4))

        # [Translated] See surrounding code for details.
        edge_lines = []
        for e_idx in connected_edges:
            pts_10 = e_feat[e_idx, 43:73].reshape(10, 3)

            if adj_FE[i, e_idx] < 0.0: # reverse points when edge orientation is REVERSED in this face
                pts_10 = pts_10[::-1]

            uv_line = []
            for pt in pts_10:
                if np.all(pt == 0): continue
                proj = GeomAPI_ProjectPointOnSurf(gp_Pnt(float(pt[0]), float(pt[1]), float(pt[2])), geom_surf)
                if proj.NbPoints() > 0:
                    u, v = proj.LowerDistanceParameters()
                    if matched_enum in [GeomAbs_Cylinder, GeomAbs_Cone, GeomAbs_Sphere, GeomAbs_Torus]:
                        if uv_line:
                            prev_u, _ = uv_line[-1]
                            while u - prev_u > math.pi: u -= 2 * math.pi
                            while prev_u - u > math.pi: u += 2 * math.pi
                        else:
                            period = 2 * math.pi
                            while u < umin - 0.1: u += period
                            while u > umax + 0.1: u -= period
                    uv_line.append([round(u, 4), round(v, 4)])

            if len(uv_line) >= 2:
                # [Translated] See surrounding code for details.
                v_indices = np.where(adj_EV[e_idx] != 0)[0]
                valid_vs = [v for v in v_indices if v in vertex_uv_map]
                if len(valid_vs) == 1: # closed edge
                    uv_line[0] = uv_line[-1] = list(vertex_uv_map[valid_vs[0]])
                elif len(valid_vs) >= 2: # regular edge
                    v1, v2 = vertex_uv_map[valid_vs[0]], vertex_uv_map[valid_vs[1]]
                    if math.hypot(uv_line[0][0]-v1[0], uv_line[0][1]-v1[1]) < math.hypot(uv_line[0][0]-v2[0], uv_line[0][1]-v2[1]):
                        uv_line[0], uv_line[-1] = list(v1), list(v2)
                    else:
                        uv_line[0], uv_line[-1] = list(v2), list(v1)
                edge_lines.append(uv_line)

        # [Translated] See surrounding code for details.
        valid_polygon = Polygon([(umin, vmin), (umax, vmin), (umax, vmax), (umin, vmax)]) # default: full UV bounding box
        
        if edge_lines:
            try:
                merged = linemerge(edge_lines)
                polys = [p.buffer(0) for p in polygonize(merged) if p.area > 1e-10]
                if polys:
                    # Even-odd fill in UV space keeps concave boundaries and interior holes.
                    parity_shape = polys[0]
                    for poly in polys[1:]:
                        parity_shape = parity_shape.symmetric_difference(poly)

                    bbox_poly = Polygon([(umin, vmin), (umax, vmin), (umax, vmax), (umin, vmax)])
                    valid_polygon = parity_shape.intersection(bbox_poly).buffer(0)
            except Exception as e:
                print(e)
                pass # translated comment

        # [Translated] See surrounding code for details.
        u_steps, v_steps = 100, 100
        u_vals, v_vals = np.linspace(umin, umax, u_steps), np.linspace(vmin, vmax, v_steps)
        f_verts, f_tris, vertex_idx_map, valid_idx = [], [], {}, 0

        for r, u in enumerate(u_vals):
            for c, v in enumerate(v_vals):
                if not valid_polygon.covers(Point(u, v)): continue
                p = geom_surf.Value(u, v)
                f_verts.append([p.X(), p.Y(), p.Z()])
                vertex_idx_map[(r, c)] = valid_idx
                valid_idx += 1

        for r in range(u_steps - 1):
            for c in range(v_steps - 1):
                if all(k in vertex_idx_map for k in [(r,c), (r+1,c), (r,c+1), (r+1,c+1)]):
                    idx1, idx2, idx3, idx4 = vertex_idx_map[(r,c)], vertex_idx_map[(r+1,c)], vertex_idx_map[(r,c+1)], vertex_idx_map[(r+1,c+1)]
                    if is_reversed: f_tris.extend([[idx1, idx2, idx3], [idx2, idx4, idx3]])
                    else: f_tris.extend([[idx1, idx3, idx2], [idx2, idx3, idx4]])

        if len(f_verts) >= 3 and len(f_tris) > 0:
            tmp_mesh = o3d.geometry.TriangleMesh()
            tmp_mesh.vertices = o3d.utility.Vector3dVector(normalize_pts(np.array(f_verts)))
            tmp_mesh.triangles = o3d.utility.Vector3iVector(np.array(f_tris))
            tmp_mesh.paint_uniform_color(face_color)
            tmp_mesh.translate(offset)
            tmp_mesh.compute_vertex_normals()
            recon_mesh += tmp_mesh

# ==========================================
# [Translated] See surrounding code for details.
# ==========================================

geometry_list = [orig_mesh, orig_wireframe, recon_mesh, recon_wireframe]
# geometry_list = [orig_mesh, orig_wireframe, recon_wireframe]

vis = o3d.visualization.Visualizer()
vis.create_window(window_name=f"Tensor Verification: Original (Left) vs Extracted (Right)", width=1600, height=900)

opt = vis.get_render_option()
opt.background_color = np.asarray([1.0, 1.0, 1.0]) # white background
opt.mesh_show_wireframe = False # show mesh wireframe overlay
opt.mesh_show_back_face = True # render back faces

for geom in geometry_list:
    vis.add_geometry(geom)

vis.run()
vis.destroy_window()

Visualizing Model ID: 00104748
